In [1]:
import sys
from pathlib import Path
# Get project root (current working directory when running from project root)
# If running from research/, use parent; otherwise use cwd()
project_root = Path.cwd() if (Path.cwd() / 'vault').exists() else Path.cwd().parent
sys.path.insert(0, str(project_root))

In [2]:
from utils.enums import Ticker, TimeFrame, Direction
from datetime import datetime
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

from feature_extraction.feature_extractor import extract_features_for_bias_node
from feature_selection.base_models import BaseModel, QuantileBinningModel
from ensemble.vault_manager import (
    initialize_vault, create_ensemble_directory, 
    load_feature_base_models, list_features
)
from eda.feature_explorer import FeatureExplorer
import utils.helpers as helpers

/home/raman/repos/Trading-Algo/utils/permutation_test/permute_bars.py:38: UserWarning: feature_selection.opt_thresh not found. Using simplified threshold optimization. For full functionality, ensure the opt_thresh module is available.
  warnings.warn(


# RSI Bias Node Test - Lookback = 2

This notebook demonstrates the simplified workflow using the new helper functions:
1. **One-line feature extraction**: Using `extract_features_for_bias_node()` to extract features with forward returns automatically
2. **Self-contained exploration**: Using `FeatureExplorer` which handles binning model fitting internally
3. **Comprehensive analysis**: Using `generate_summary_report()` to run all relevant analysis methods at once
4. **Vault operations**: Creating and saving BaseModel to vault for production use

## Key Improvements:
- **Simplified workflow**: No manual forward returns calculation or BaseModel fitting for exploration
- **Automatic alignment**: Features and targets are automatically aligned
- **Self-contained**: FeatureExplorer handles model fitting internally
- **Comprehensive**: Single method call generates full analysis report

In [3]:
# Step 1: Extract features with forward returns automatically
# The new helper function handles everything: feature extraction + forward returns computation
# Define tickers for this ensemble (used throughout notebook)
ensemble_tickers = [Ticker.NQ, Ticker.ES, Ticker.RTY, Ticker.YM]

#Define bias node specification
bias_spec = {
    'module_name': 'williamsr',
    'timeframes': [TimeFrame.D],
    'params': {'lookback': [2,3,4,5,6,7,8,9,10,12,14,16]}
}

# bias_spec = {
#     'module_name': 'prev_return',
#     'timeframes': [TimeFrame.D],
#     'params': {}
# }

# Extract features with forward returns in one call!
# This automatically:
# 1. Loads candles for all tickers
# 2. Computes forward returns (avoiding lookahead bias)
# 3. Extracts features
# 4. Aligns features and targets correctly
print("Extracting features with forward returns...")
features_df, targets_df = extract_features_for_bias_node(
    bias_spec=bias_spec,
    ticker=ensemble_tickers,
    start=datetime(2000, 1, 1),
    end=datetime(2024, 12, 31),
    use_millisecond_offset=True, 
    target_col='log_return_atr'
)

print(f"\n✓ Features extracted successfully!")
print(f"  Features shape: {features_df.shape}")
print(f"  Targets shape: {targets_df.shape}")
print(f"  Feature columns: {[col for col in features_df.columns if col != 'ticker']}")
print(f"  Target columns: {[col for col in targets_df.columns if col != 'ticker']}")
print(f"  Unique tickers: {features_df['ticker'].unique() if 'ticker' in features_df.columns else 'N/A'}")
print(f"\n✓ Forward returns calculated correctly:")
print(f"  - Return at timestamp t = return from t to t+1")
print(f"  - Feature at timestamp t predicts return[t] (forward return)")
print(f"  - No lookahead bias: feature uses data up to t, predicts return from t to t+1")

Extracting features with forward returns...

✓ Features extracted successfully!
  Features shape: (20294, 13)
  Targets shape: (20294, 5)
  Feature columns: ['williamsr_signal_D_atrLength_14_compression_1.0_lookback_2', 'williamsr_signal_D_atrLength_14_compression_1.0_lookback_3', 'williamsr_signal_D_atrLength_14_compression_1.0_lookback_4', 'williamsr_signal_D_atrLength_14_compression_1.0_lookback_5', 'williamsr_signal_D_atrLength_14_compression_1.0_lookback_6', 'williamsr_signal_D_atrLength_14_compression_1.0_lookback_7', 'williamsr_signal_D_atrLength_14_compression_1.0_lookback_8', 'williamsr_signal_D_atrLength_14_compression_1.0_lookback_9', 'williamsr_signal_D_atrLength_14_compression_1.0_lookback_10', 'williamsr_signal_D_atrLength_14_compression_1.0_lookback_12', 'williamsr_signal_D_atrLength_14_compression_1.0_lookback_14', 'williamsr_signal_D_atrLength_14_compression_1.0_lookback_16']
  Target columns: ['raw_return', 'log_return', 'log_return_atr', 'log_return_ewsd']
  Unique t

In [4]:
# Step 2: Create FeatureExplorer for analysis
# Build metadata for FeatureExplorer (optional but recommended)
metadata = {'feature_metadata': {}}
for feature_col in features_df.columns:
    if feature_col != 'ticker':
        parsed = helpers.parse_feature_column_name(feature_col)
        metadata['feature_metadata'][feature_col] = {
            'module': parsed.get('module'),
            'parameters': parsed.get('params', {}),
            'timeframes': [parsed.get('tf')] if parsed.get('tf') else [],
            'base_name': parsed.get('module'),
            'full_name': feature_col
        }

# Create FeatureExplorer (self-contained - no BaseModel needed!)
explorer = FeatureExplorer(
    features_df=features_df,
    targets_df=targets_df,
    metadata=metadata
)

print("FeatureExplorer created successfully!")
print(f"  Number of features: {explorer.n_features}")
print(f"  Feature names: {explorer.feature_names}")
print(f"  Number of samples: {explorer.n_samples}")
print(f"  Date range: {explorer.date_range[0].date()} to {explorer.date_range[1].date()}")


FeatureExplorer created successfully!
  Number of features: 12
  Feature names: ['williamsr_signal_D_atrLength_14_compression_1.0_lookback_2', 'williamsr_signal_D_atrLength_14_compression_1.0_lookback_3', 'williamsr_signal_D_atrLength_14_compression_1.0_lookback_4', 'williamsr_signal_D_atrLength_14_compression_1.0_lookback_5', 'williamsr_signal_D_atrLength_14_compression_1.0_lookback_6', 'williamsr_signal_D_atrLength_14_compression_1.0_lookback_7', 'williamsr_signal_D_atrLength_14_compression_1.0_lookback_8', 'williamsr_signal_D_atrLength_14_compression_1.0_lookback_9', 'williamsr_signal_D_atrLength_14_compression_1.0_lookback_10', 'williamsr_signal_D_atrLength_14_compression_1.0_lookback_12', 'williamsr_signal_D_atrLength_14_compression_1.0_lookback_14', 'williamsr_signal_D_atrLength_14_compression_1.0_lookback_16']
  Number of samples: 20294
  Date range: 2000-01-03 to 2024-12-30


In [5]:
# Step 6b: Generate comprehensive summary report with EXPORT
# This runs all relevant analysis methods and exports to files
# First, create and configure the binning model
binning_model = QuantileBinningModel(n_bins=10, selection_metric='sortino', strategy='long')

print("\nGenerating comprehensive summary report with export...")
results = explorer.generate_summary_report(
    binning_model=binning_model,
    target_col='log_return',
    strategy='long',
    show_plots=False,  # Set to False when exporting
    verbose=True,
    export_report=True,  # Export report to files
    export_path='./reports/summary_report',  # Export directory
    permutation_test_nreps=100,
)

print("\nSummary Report Complete!")
print("  (Report exported to files - see export_path for details)")
print(f"  - Summary stats: {len(results['summary_stats'])} features")
print(f"  - Feature-target correlations: {len(results['correlations'])} features")
print(f"  - Intra-feature correlations: {len(results.get('feature_correlations', pd.DataFrame()).columns)} features")
print(f"  - Signal cumsum plots: {len(results['signal_cumsum_figures'])} features")
if results.get('parameter_sensitivity'):
    print(f"  - Parameter sensitivity analyses: {len(results['parameter_sensitivity'])}")
if results.get('permutation_test') is not None:
    n_significant = results['permutation_test']['significant'].sum() if 'significant' in results['permutation_test'].columns else 0
    print(f"  - Permutation test: {len(results['permutation_test'])} features tested, {n_significant} significant")
    print(f"\n  Permutation test p-values:")
    for _, row in results['permutation_test'].iterrows():
        pval = row.get('pval', 'N/A')
        sig = '✓' if row.get('significant', False) else '✗'
        print(f"    {sig} {row.get('feature', 'unknown')}: p = {pval:.4f}")



Generating comprehensive summary report with export...

Generating Comprehensive Feature Analysis Report

[1/10] Computing basic summary statistics...
  ✓ Computed statistics for 12 features

[2/10] Computing feature-target correlations...
  ✓ Computed correlations for 12 features

[3/10] Computing intra-feature correlations...
  ✓ Computed correlation matrix for 12 features

[4/10] Plotting decile analysis...
  ✓ Generated 12 decile plots

[5/10] Plotting signal-gated cumulative returns...
  ✓ Generated 12 signal plots

[6/10] Analyzing parameter sensitivity...
[DEBUG] analyze_parameter: 'williamsr_signal_D_atrLength_14_compression_1.0_lookback_2' samples=20294 (feature NaNs=0, target NaNs=0)
[DEBUG] analyze_parameter: 'williamsr_signal_D_atrLength_14_compression_1.0_lookback_3' samples=20294 (feature NaNs=0, target NaNs=0)
[DEBUG] analyze_parameter: 'williamsr_signal_D_atrLength_14_compression_1.0_lookback_4' samples=20294 (feature NaNs=0, target NaNs=0)
[DEBUG] analyze_parameter: '

/home/raman/repos/Trading-Algo/metrics/plotting/distribution.py:74: RuntimeWarning:

More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.



  ✓ Generated 12 distribution plots

[9/10] Plotting time series...
  ✓ Generated 12 time series plots

[10/10] Running permutation test...
  ✓ Completed permutation test for 12 features
  ✓ Found 12 significant features (p <= 0.1)

  Permutation test p-values:
    ✓ williamsr_signal_D_atrLength_14_compression_1.0_lookback_3: p = 0.0100
    ✓ williamsr_signal_D_atrLength_14_compression_1.0_lookback_10: p = 0.0100
    ✓ williamsr_signal_D_atrLength_14_compression_1.0_lookback_6: p = 0.0100
    ✓ williamsr_signal_D_atrLength_14_compression_1.0_lookback_8: p = 0.0100
    ✓ williamsr_signal_D_atrLength_14_compression_1.0_lookback_9: p = 0.0100
    ✓ williamsr_signal_D_atrLength_14_compression_1.0_lookback_7: p = 0.0100
    ✓ williamsr_signal_D_atrLength_14_compression_1.0_lookback_16: p = 0.0100
    ✓ williamsr_signal_D_atrLength_14_compression_1.0_lookback_12: p = 0.0200
    ✓ williamsr_signal_D_atrLength_14_compression_1.0_lookback_5: p = 0.0300
    ✓ williamsr_signal_D_atrLength_14_comp

In [ ]:

# Step 7: Create BaseModel for vault operations (if needed)
# BaseModel is only needed when saving to vault
print("\n" + "="*70)
print("Creating BaseModel for vault operations...")
print("="*70)

binning_model_vault = QuantileBinningModel(n_bins=10, selection_metric='sortino', strategy='long')

base_model = BaseModel(
    feature_config={'bias_node_spec': bias_spec},
    binning_model=binning_model_vault,
    tickers=ensemble_tickers
)

print("BaseModel created for vault operations!")
print(f"  Ticker: {base_model.ticker.name}")
print(f"  Bias node spec: {base_model.bias_node_spec}")
print(f"  Number of bias nodes: {len(base_model.bias_nodes)}")

# Load candles for BaseModel fitting (needed for vault save)
candles_df = helpers.load_data_multi_ticker(
    tickers=ensemble_tickers,
    timeframe=TimeFrame.D,
    start=datetime(2000, 1, 1),
    end=datetime(2024, 12, 31),
    use_millisecond_offset=True
)

# Compute forward returns for BaseModel
from feature_extraction.feature_extractor import compute_forward_returns
targets_for_base = compute_forward_returns(candles_df)

# Filter candles to match targets
target_datetimes = pd.DatetimeIndex(targets_for_base.index.unique())
all_candles = candles_df[candles_df['datetime'].isin(target_datetimes)].copy()
all_candles_fit = all_candles.reset_index(drop=True)

target_series = pd.Series(
    targets_for_base['log_return'].values,
    index=pd.DatetimeIndex(targets_for_base.index),
    name='log_return'
)

# Fit BaseModel (needed for vault save)
base_model.fit(all_candles_fit, target_series)
print(f"✓ BaseModel fitted successfully!")
print(f"  Feature column: {base_model.feature_column}")
print(f"  Best long bin: {base_model.binning_model.best_long_bin_}")

print("\nCreating ensemble directory...")
ensemble_dir = create_ensemble_directory(
    timeframe=TimeFrame.D,
    ensemble_name='rsi_bias_lookback_2',
    direction=Direction.LONG,
    tickers=ensemble_tickers
)

print(f"✓ Created ensemble directory: {ensemble_dir}")
print(f"  Ensemble tickers: {[t.name for t in ensemble_tickers]}")
print(f"  Default ensemble directory is now set automatically")

In [ ]:
# Step 8: Save base model to vault
print("\nSaving base model to vault...")
# Pass all ensemble tickers when saving
# ensemble_dir is optional - uses default set by create_ensemble_directory()
model_id = base_model.save_to_vault(tickers=ensemble_tickers)
print(f"✓ Saved model with ID: {model_id}")
print(f"  Feature column: {base_model.feature_column}")
print(f"  Model ID: {model_id}")
print(f"  Saved with tickers: {[t.name for t in ensemble_tickers]}")

In [ ]:
# Step 9: Verify vault save/load
print("\nLoading models from vault...")
# ensemble_dir is optional - uses default set by create_ensemble_directory()
# Path resolution automatically handles running from research/ directory
loaded_models = load_feature_base_models(feature_column=base_model.feature_column)
model_id = 'quantile_binning_2'

print(f"✓ Loaded {len(loaded_models)} model(s)")
print(f"  Model keys (ticker, model_id): {list(loaded_models.keys())}")

# Verify loaded model (keys are now (ticker, model_id) tuples)
key = (Ticker.ES, model_id)
if key in loaded_models:
    loaded_model = loaded_models[key]
    print(f"\n✓ Model '{model_id}' for {Ticker.ES.name} loaded successfully")
    print(f"  Binning model fitted: {loaded_model.binning_model.is_fitted_}")
    print(f"  Best long bin: {loaded_model.binning_model.best_long_bin_}")
    print(f"  Ticker: {loaded_model.ticker.name}")
else:
    print(f"\n✗ Model '{model_id}' not found. Available keys: {list(loaded_models.keys())}")

# List all features in ensemble
print("\n" + "="*70)
print("Features in ensemble:")
print("="*70)
# ensemble_dir is optional - uses default set by create_ensemble_directory()
features_list = list_features()
print(features_list)